# Silver Layer: Incremental MERGE (Upsert)
Transactional tables: orders, order_items, payments.
Instead of overwriting, we MERGE on the business key so re-running never duplicates rows.
- orders, payments: update when the incoming row is newer (upsert)
- order_items: insert-only (a line item never changes once written)

In [0]:
spark.sql("DROP TABLE IF EXISTS atliq.silver.orders")
spark.sql("DROP TABLE IF EXISTS atliq.silver.payments")
spark.sql("DROP TABLE IF EXISTS atliq.silver.order_items")
print("dropped stale merge tables")

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

STORAGE_ACCOUNT = "atliqlakeab"
BRONZE = f"abfss://lakehouse@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"

def read_all(table):
    # recursiveFileLookup walks all subfolders (incl. ingest_date=...) and reads every parquet file
    return (spark.read
            .option("recursiveFileLookup", "true")
            .parquet(f"{BRONZE}/{table}"))

## Orders
Deduplicate to the latest row per order_id, then MERGE. Update only when the incoming row is newer (updated_at), else insert.

In [0]:
w_ord = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
orders = (
    read_all("orders")
    .withColumn("rn", F.row_number().over(w_ord)).filter("rn = 1").drop("rn")
    .withColumn("order_date",   F.to_date("order_date"))
    .withColumn("status",       F.initcap(F.trim("status")))
    .withColumn("order_amount", F.col("order_amount").cast("decimal(12,2)"))
    .filter(F.col("order_id").isNotNull())
)
(DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.orders").addColumns(orders.schema).execute())
(DeltaTable.forName(spark, "atliq.silver.orders").alias("t")
    .merge(orders.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute())
print("orders silver count ->", spark.table("atliq.silver.orders").count())

## Payments
Same upsert pattern on payment_id: update when newer, else insert.

In [0]:
w_pay = Window.partitionBy("payment_id").orderBy(F.col("updated_at").desc())
payments = (
    read_all("payments")
    .withColumn("rn", F.row_number().over(w_pay)).filter("rn = 1").drop("rn")
    .withColumn("amount",  F.col("amount").cast("decimal(12,2)"))
    .withColumn("paid_at", F.col("paid_at").cast("timestamp"))
    .filter(F.col("payment_id").isNotNull())
)
(DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.payments").addColumns(payments.schema).execute())
(DeltaTable.forName(spark, "atliq.silver.payments").alias("t")
    .merge(payments.alias("s"), "t.payment_id = s.payment_id")
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute())
print("payments silver count ->", spark.table("atliq.silver.payments").count())

## Order Items
Insert-only: a line item is immutable once created, so we only insert new order_item_ids, never update.

In [0]:
w_it = Window.partitionBy("order_item_id").orderBy(F.col("created_at").desc())
order_items = (
    read_all("order_items")
    .withColumn("rn", F.row_number().over(w_it)).filter("rn = 1").drop("rn")
    .withColumn("quantity",   F.col("quantity").cast("int"))
    .withColumn("item_price", F.col("item_price").cast("decimal(10,2)"))
    .filter(F.col("order_item_id").isNotNull())
)
(DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.order_items").addColumns(order_items.schema).execute())
(DeltaTable.forName(spark, "atliq.silver.order_items").alias("t")
    .merge(order_items.alias("s"), "t.order_item_id = s.order_item_id")
    .whenNotMatchedInsertAll()
    .execute())
print("order_items silver count ->", spark.table("atliq.silver.order_items").count())

### MERGE proof

In [0]:
%sql DESCRIBE HISTORY atliq.silver.orders

### reconciliation in Silver

In [0]:
%sql
SELECT
  (SELECT SUM(order_amount)          FROM atliq.silver.orders)      AS header_total,
  (SELECT SUM(quantity * item_price) FROM atliq.silver.order_items) AS line_item_total,
  (SELECT SUM(amount)                FROM atliq.silver.payments)    AS payment_total